# xfig_08 — Night Fingerprint Heatmap

For a few representative subjects, show a 2D heatmap:
- **rows** = context lengths (30s → 240m)
- **columns** = normalised window position in the night (0% → 100%)
- **color** = predicted probability of positive class

Shows how the model's certainty changes as a function of BOTH where in
the night a window falls AND how much context the model has.

Idea #8 from `docs/NEW_PLOT_IDEAS.md`.

**Data**: `collected/predictions/*.parquet` (needs `window_idx` column).

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
EXPLORE_DIR    = NSRR_TOOLS / "results" / "paper_figures" / "explore"
FINAL_OUT      = EXPLORE_DIR / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)
TABLES_DIR     = NSRR_TOOLS / "results" / "tables"

# Add explore utils to path
_nb_dir = EXPLORE_DIR / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.data_explore import (
    set_root, load_analysis, load_analysis_all_k,
    load_heatmap, load_parquets, load_modality_table,
    subject_predictions, subject_correctness_matrix, CONTEXT_TO_MIN, CTX_ORDER,
)
from utils import panels_explore as xp

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

set_root(WORKSPACE_ROOT)
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 7,
})

# ── Constants ──────────────────────────────────────────────────────────────────
MAIN_TASKS = ["sex_binary", "bmi_binary", "age_class",
              "sleep_efficiency_binary", "apnea_binary"]
TASK_LABEL = xp.TASK_LABEL

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND'}")


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
TASK  = "sex_binary"         # change to any binary task
HEAD  = "transformer"
SPLIT = "test"
CONTEXTS = ["30s", "10m", "40m", "80m", "120m", "240m"]
N_BINS = 25   # night position bins

pqs = load_parquets("phase0_v3", TASK, HEAD, SPLIT)
print("Contexts loaded:", list(pqs.keys()))
if pqs:
    sample = list(pqs.values())[0]
    print("Columns:", sample.columns.tolist())
    print("window_idx present:", "window_idx" in sample.columns)

In [ ]:
# ── Pick 4 representative subjects ────────────────────────────────────────────
reps = xp.pick_representative_subjects(pqs, n_per_type=1)
print("Representative subjects:", reps)

# Flatten to a list: [always_correct, always_wrong, context_sensitive_pos, context_sensitive_neg]
SUBJECT_TYPES = {
    "Always correct":          "always_correct",
    "Always wrong":            "always_wrong",
    "Improves with context":   "context_sensitive_pos",
    "Worsens with context":    "context_sensitive_neg",
}
subjects_to_plot = []
titles = []
for title, key in SUBJECT_TYPES.items():
    ids = reps.get(key, [])
    if ids:
        subjects_to_plot.append(ids[0])
        titles.append(title)

In [ ]:
if not subjects_to_plot:
    print("No subjects found — check that parquets have window_idx column")
else:
    n_sub = len(subjects_to_plot)
    fig, axes = plt.subplots(1, n_sub, figsize=(3.5 * n_sub, 3.5))
    if n_sub == 1:
        axes = [axes]

    for ax, subj, title in zip(axes, subjects_to_plot, titles):
        xp.night_fingerprint_panel(ax, pqs, subj,
                                    contexts=CONTEXTS, n_bins=N_BINS)
        ax.set_title(f"{title}\n(ID: {subj[:8]}…)", fontsize=7)

    fig.suptitle(
        f"Night fingerprint — {TASK_LABEL.get(TASK, TASK)} / {HEAD.upper()}",
        fontsize=8, y=1.02,
    )
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
fig.savefig(str(FINAL_OUT / 'xfig_08_night_fingerprint.pdf'), bbox_inches='tight')
fig.savefig(str(FINAL_OUT / 'xfig_08_night_fingerprint.png'), dpi=150, bbox_inches='tight')
print('Saved →', FINAL_OUT / 'xfig_08_night_fingerprint.pdf')